# Financial Market Data Collection & Preprocessing

## Objective

The objective of this notebook is to build a clean and structured financial dataset for future quantitative finance research tasks including:

- Cointegration analysis
- Engle-Granger testing
- Johansen testing
- Spread analysis
- Rolling window stability analysis
- Statistical arbitrage research

The dataset consists of major US equities from the S&P 500 along with sector and market ETFs downloaded using Yahoo Finance.

## Import Libraries 

In [4]:
import yfinance as yf
import pandas as pd
import numpy as np

import requests
from bs4 import BeautifulSoup
from io import StringIO

In [34]:
# stocks = [

#     # Technology
#     'AAPL','MSFT','NVDA','GOOG','META','AMZN','TSLA',
#     'AMD','INTC','CSCO','ORCL','CRM','ADBE','QCOM',
#     'TXN','AVGO','IBM','AMAT','MU','NOW',

#     # Banking & Finance
#     'JPM','BAC','WFC','GS','MS','C','AXP','BLK',
#     'SCHW','USB','PNC','TFC',

#     # Energy
#     'XOM','CVX','COP','SLB','EOG','PSX','MPC',

#     # Healthcare
#     'JNJ','PFE','MRK','ABBV','TMO','LLY','ABT',
#     'DHR','BMY','UNH','CVS',

#     # Consumer
#     'KO','PEP','PG','COST','WMT','HD','MCD',
#     'SBUX','NKE','LOW','TGT',

#     # Industrials
#     'CAT','BA','HON','GE','MMM','UPS','FDX',
#     'LMT','DE',

#     # Telecom & Media
#     'VZ','T','CMCSA','DIS','NFLX',

#     # Utilities
#     'NEE','DUK','SO',

#     # Real Estate
#     'AMT','PLD','SPG',

#     # Materials
#     'LIN','APD','FCX',

#     # Transportation
#     'UNP','CSX',

#     # Semiconductor
#     'NXPI','KLAC','LRCX',

#     # Additional large caps
#     'PYPL','UBER','SHOP','ZM'
# ]

## Retrieve top 100 S&P 500 Companies 
The list of companies was extracted from the S&P 500 Wikipedia dataset instead of manually specifying individual stock names. This approach ensures a more systematic, scalable, and reproducible selection of top US market companies.


In [5]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table', {'id': 'constituents'})

sp500 = pd.read_html(
    StringIO(str(table))
)[0]

sp500.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [ ]:
import yfinance as yf
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

symbols = [s.replace('.', '-') for s in sp500['Symbol']]

def get_market_cap(symbol):
    try:
        ticker = yf.Ticker(symbol)
        # Fast history call or fast info if available, but standard info works better in parallel
        return {"Symbol": symbol, "MarketCap": ticker.info.get("marketCap", None)}
    except Exception:
        return {"Symbol": symbol, "MarketCap": None}

market_caps = []

# Use ThreadPoolExecutor to make multiple network requests at the same time
# max_workers=20 is a safe sweet spot to avoid getting blocked by Yahoo
with ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(get_market_cap, symbol) for symbol in symbols]
    
    for future in as_completed(futures):
        market_caps.append(future.result())


market_cap_df = pd.DataFrame(market_caps)

In [43]:
market_cap_df.head()

market_cap_df.shape

(503, 2)

In [40]:
market_cap_df = market_cap_df.dropna()

top100 = (
    market_cap_df
    .sort_values("MarketCap", ascending=False)
    .head(100)
)

top100.head(20)

,Symbol,MarketCap
342,NVDA,5189349146624
6,GOOGL,4726597877760
8,GOOG,4678015254528
28,AAPL,4589945880576
305,MSFT,3171867426816
16,AMZN,2947448045568
81,AVGO,2019714793472
433,TSLA,1660405547008
307,META,1612634914816
323,MU,1041484939264


In [44]:
top100.to_csv(
    "../data/raw/top100_us_stocks.csv",
    index=False
)

In [45]:
stocks = top100["Symbol"].tolist()

print(len(stocks))

100


## Selection and Formatting of Stock Symbols

The stock symbols of the first 100 S&P 500 companies are extracted from the dataset to create the research universe for analysis. Certain ticker symbols containing periods (.) are converted to hyphen (-) format to ensure compatibility with Yahoo Finance data retrieval conventions.


In [49]:
stocks = top100["Symbol"].tolist()



stocks = [stock.replace('.', '-') for stock in stocks]

stocks[:30]

['NVDA',
 'GOOGL',
 'GOOG',
 'AAPL',
 'MSFT',
 'AMZN',
 'AVGO',
 'TSLA',
 'META',
 'MU',
 'BRK-B',
 'LLY',
 'WMT',
 'AMD',
 'JPM',
 'V',
 'XOM',
 'INTC',
 'ORCL',
 'JNJ',
 'CSCO',
 'COST',
 'MA',
 'CAT',
 'LRCX',
 'ABBV',
 'CVX',
 'NFLX',
 'BAC',
 'AMAT']

In [53]:
print("Total symbols:", len(top100["Symbol"]))
print("Unique symbols:", top100["Symbol"].nunique())

Total symbols: 100
Unique symbols: 100


## Add Major ETFs

Major ETFs are manually included to provide representation of broad market indices and sector-specific benchmarks that are not part of the S&P 500 company list. These ETFs help capture overall market behavior, sector movements, and diversified market exposure for future correlation and cointegration analysis.


In [50]:
etfs = [
    'SPY','QQQ','DIA','IWM',
    'XLF','XLK','XLE','XLV',
    'XLY','XLP','XLI','XLU',
    'VNQ','GLD','TLT'
]

In [51]:
symbols = stocks + etfs

len(symbols)

115

## Historical Market Data Collection

Historical price data is downloaded using the Yahoo Finance API through the yfinance library. The dataset includes daily market data from 2018 onward, while the end date is dynamically generated.
The dataset begins in 2018 to ensure sufficient historical coverage across multiple market regimes, including:

- Bull market periods
- COVID-19 market crash
- Recovery phases
- Inflationary environments


In [54]:
print("AXP" in symbols)
print("BRK-B" in symbols)

True
True


In [52]:
start_date = '2018-01-01'

# Dynamic latest date
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

data = yf.download(
    symbols,
    start=start_date,
    end=end_date,
    auto_adjust=False
)

[**********************83%***************        ]  96 of 115 completed$AXP: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-05-29)
[**********************94%********************   ]  108 of 115 completed$BRK-B: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-05-29)
[*********************100%***********************]  115 of 115 completed

2 Failed downloads:
['AXP', 'BRK-B']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-05-29)


In [57]:
import yfinance as yf
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

axp = yf.download(
    "AXP",
    start="2018-01-01",
    end=end_date,
    auto_adjust=False
)

print(axp.shape)
axp.head()

[*********************100%***********************]  1 of 1 completed

(2112, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,AXP,AXP,AXP,AXP,AXP,AXP
2018-01-02,88.292412,98.940002,99.730003,98.220001,99.730003,2746700
2018-01-03,88.836769,99.550003,99.760002,99.019997,99.239998,2976400
2018-01-04,90.314415,100.849998,101.650002,99.910004,100.029999,3122000
2018-01-05,90.520355,101.080002,101.080002,100.269997,100.849998,2550300
2018-01-08,89.687523,100.150002,101.199997,100.120003,101.000000,1896500


In [58]:
brk = yf.download(
    "BRK-B",
    start="2018-01-01",
    end=end_date,
    auto_adjust=False
)

print(brk.shape)
brk.head()

[*********************100%***********************]  1 of 1 completed

(2112, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,BRK-B,BRK-B,BRK-B,BRK-B,BRK-B,BRK-B
2018-01-02,197.220001,197.220001,198.869995,195.960007,198.869995,4113000
2018-01-03,199.789993,199.789993,200.000000,197.000000,197.000000,3526700
2018-01-04,200.690002,200.690002,202.130005,200.009995,200.899994,3900000
2018-01-05,201.419998,201.419998,201.669998,199.309998,201.000000,4207600
2018-01-08,202.740005,202.740005,203.070007,199.800003,201.149994,3887900


In [64]:
axp.shape
brk.shape

(2112, 6)

In [65]:
print(axp.columns)
print()
print(brk.columns)

MultiIndex([('Adj Close', 'AXP'),
            (    'Close', 'AXP'),
            (     'High', 'AXP'),
            (      'Low', 'AXP'),
            (     'Open', 'AXP'),
            (   'Volume', 'AXP')],
           names=['Price', 'Ticker'])

MultiIndex([('Adj Close', 'BRK-B'),
            (    'Close', 'BRK-B'),
            (     'High', 'BRK-B'),
            (      'Low', 'BRK-B'),
            (     'Open', 'BRK-B'),
            (   'Volume', 'BRK-B')],
           names=['Price', 'Ticker'])


In [66]:
print(type(axp.columns))
print(type(data.columns))

<class 'pandas.MultiIndex'>
<class 'pandas.MultiIndex'>


In [67]:
axp.head()

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,AXP,AXP,AXP,AXP,AXP,AXP
2018-01-02,88.292412,98.940002,99.730003,98.220001,99.730003,2746700
2018-01-03,88.836769,99.550003,99.760002,99.019997,99.239998,2976400
2018-01-04,90.314415,100.849998,101.650002,99.910004,100.029999,3122000
2018-01-05,90.520355,101.080002,101.080002,100.269997,100.849998,2550300
2018-01-08,89.687523,100.150002,101.199997,100.120003,101.000000,1896500


In [68]:
print(axp.columns)
print(brk.columns)

MultiIndex([('Adj Close', 'AXP'),
            (    'Close', 'AXP'),
            (     'High', 'AXP'),
            (      'Low', 'AXP'),
            (     'Open', 'AXP'),
            (   'Volume', 'AXP')],
           names=['Price', 'Ticker'])
MultiIndex([('Adj Close', 'BRK-B'),
            (    'Close', 'BRK-B'),
            (     'High', 'BRK-B'),
            (      'Low', 'BRK-B'),
            (     'Open', 'BRK-B'),
            (   'Volume', 'BRK-B')],
           names=['Price', 'Ticker'])


In [69]:
axp[('Adj Close', 'AXP')].head()

2018-01-02    88.292412
2018-01-03    88.836769
2018-01-04    90.314415
2018-01-05    90.520355
2018-01-08    89.687523
Name: (Adj Close, AXP), dtype: float64

In [70]:
brk[('Adj Close', 'BRK-B')].head()

2018-01-02    197.220001
2018-01-03    199.789993
2018-01-04    200.690002
2018-01-05    201.419998
2018-01-08    202.740005
Name: (Adj Close, BRK-B), dtype: float64

In [77]:
adj_close = data['Adj Close'].copy()

In [78]:
adj_close.loc[:, 'AXP'] = axp[('Adj Close', 'AXP')]

adj_close.loc[:, 'BRK-B'] = brk[('Adj Close', 'BRK-B')]

In [79]:
adj_close[['AXP', 'BRK-B']].isnull().sum()

Ticker
AXP      0
BRK-B    0
dtype: int64

In [80]:
adj_close[['AXP', 'BRK-B']].head()

Ticker,AXP,BRK-B
2018-01-02,88.292412,197.220001
2018-01-03,88.836769,199.789993
2018-01-04,90.314415,200.690002
2018-01-05,90.520355,201.419998
2018-01-08,89.687523,202.740005


In [81]:
adj_close[['MSFT','NVDA','META','AXP','BRK-B']].isnull().sum()

Ticker
MSFT     0
NVDA     0
META     0
AXP      0
BRK-B    0
dtype: int64

## Extraction of Adjusted Closing Prices

Adjusted closing prices are extracted from the dataset as they account for stock splits and dividends, making them more reliable for accurate financial and time series analysis.


In [82]:
adj_close.head()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
2018-01-02,40.267067,68.773148,50.506081,77.602257,48.314201,10.98,136.742584,59.450500,14.439375,20.302280,...,28.800238,25.747261,23.884140,66.254852,29.872921,45.314575,20.132584,72.723335,46.275246,58.185486
2018-01-03,40.260071,69.849373,50.617760,78.564995,49.170433,11.55,139.322906,60.209999,14.725000,20.565889,...,29.051456,26.132864,24.012455,66.611717,30.122097,45.298531,19.974421,73.419174,46.487705,59.328247
2018-01-04,40.447071,69.451019,50.531857,78.479034,49.452816,12.12,138.735733,60.479500,14.543125,20.494209,...,29.077747,26.290602,24.234871,67.099136,30.274359,45.426769,19.808548,73.523560,46.640118,59.410370
2018-01-05,40.907574,70.660042,50.677906,78.797089,49.735203,11.88,139.562393,61.457001,14.798125,20.743938,...,29.250109,26.280079,24.303314,67.560448,30.592754,45.627129,19.800833,74.149803,47.009613,59.362476
2018-01-08,40.755630,69.527908,50.531857,78.934631,50.937592,12.28,139.523743,62.343498,15.691250,20.965919,...,29.682444,26.437826,24.269094,67.838974,30.708118,45.739334,19.985998,73.880173,47.065033,59.629341


## Verification of Downloaded Symbols

The downloaded symbols are verified against the expected symbol list to identify missing or failed ticker downloads and ensure dataset completeness.


In [83]:
print("Expected Symbols:", len(symbols))

print("Downloaded Symbols:", len(adj_close.columns))

missing_symbols = set(symbols) - set(adj_close.columns)

if missing_symbols:
    print("Warning: Missing symbols detected")
    print(missing_symbols)
else:
    print("All symbols downloaded successfully")

Expected Symbols: 115
Downloaded Symbols: 115
All symbols downloaded successfully


## Save Raw Market Dataset

The complete raw market dataset is saved to preserve all downloaded fields for future research tasks including volatility analysis, liquidity analysis, and spread studies.

In [84]:
print(adj_close.shape)

adj_close[['MSFT','NVDA','META','AXP','BRK-B']].isnull().sum()

(2112, 115)


Ticker
MSFT     0
NVDA     0
META     0
AXP      0
BRK-B    0
dtype: int64

In [85]:
data.to_csv(
    '../data/raw/full_market_data.csv'
)

## Verify Datetime Index

The index is converted to datetime format to support rolling-window analysis, plotting, and statistical testing.

In [86]:
adj_close.index = pd.to_datetime(adj_close.index)

adj_close.index

DatetimeIndex(['2018-01-02', '2018-01-03', '2018-01-04', '2018-01-05',
               '2018-01-08', '2018-01-09', '2018-01-10', '2018-01-11',
               '2018-01-12', '2018-01-16',
               ...
               '2026-05-14', '2026-05-15', '2026-05-18', '2026-05-19',
               '2026-05-20', '2026-05-21', '2026-05-22', '2026-05-26',
               '2026-05-27', '2026-05-28'],
              dtype='datetime64[s]', length=2112, freq=None)

## Check for Duplicate Dates

Duplicate dates are identified and removed to maintain dataset consistency and ensure accurate time series analysis.


In [87]:
print(
    "Duplicate Dates:",
    adj_close.index.duplicated().sum()
)

adj_close = adj_close[
    ~adj_close.index.duplicated()
]

Duplicate Dates: 0


## Check for any missing value


In [88]:
adj_close.isnull().sum()

Ticker
AAPL    0
ABBV    0
ABT     0
ADI     0
AMAT    0
       ..
XLP     0
XLU     0
XLV     0
XLY     0
XOM     0
Length: 115, dtype: int64

## Missing value Percentage


In [89]:
missing_pct = (
    adj_close.isnull().mean() * 100
)

missing_pct.sort_values(
    ascending=False
)

Ticker
SNDK    84.706439
GEV     74.242424
APP     39.109848
PLTR    32.717803
CRWD    17.140152
          ...    
ETN      0.000000
DIS      0.000000
DIA      0.000000
DHR      0.000000
XOM      0.000000
Length: 115, dtype: float64

## Filtering Assets with Excessive Missing Values

Assets containing more than 5% missing observations are removed to maintain dataset consistency and improve the reliability of future financial analysis.
A 5% threshold is used to balance data completeness.

In [90]:
valid_cols = missing_pct[
    missing_pct < 5
].index

adj_close = adj_close[valid_cols]

## Handling Remaining Missing Values

Remaining missing values are filled using forward-fill and backward-fill methods to create a complete dataset for time series analysis.


In [91]:
adj_close = adj_close.ffill().bfill()

In [92]:
# Final check for any remaining missing values
adj_close.isnull().sum().sum()

0

## Validation check 
check fir any zero or negative price values

In [93]:
(adj_close <= 0).sum().sum()

0

In [94]:
# download the cleaned adjusted close data
adj_close.to_csv(
    '../data/cleaned/adj_close_cleaned.csv'
)

In [95]:
# Calculate daily returns
returns = adj_close.pct_change().dropna()

returns.head()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
2018-01-03,-0.000174,0.015649,0.002211,0.012406,0.017722,0.051913,0.018870,0.012775,0.019781,0.012984,...,0.008723,0.014976,0.005372,0.005386,0.008341,-0.000354,-0.007856,0.009568,0.004591,0.019640
2018-01-04,0.004645,-0.005703,-0.001697,-0.001094,0.005743,0.049351,-0.004214,0.004476,-0.012351,-0.003485,...,0.000905,0.006036,0.009263,0.007317,0.005055,0.002831,-0.008304,0.001422,0.003279,0.001384
2018-01-05,0.011385,0.017408,0.002890,0.004053,0.005710,-0.019802,0.005959,0.016163,0.017534,0.012185,...,0.005928,-0.000400,0.002824,0.006875,0.010517,0.004411,-0.000389,0.008518,0.007922,-0.000806
2018-01-08,-0.003714,-0.016022,-0.002882,0.001746,0.024176,0.033670,-0.000277,0.014425,0.060354,0.010701,...,0.014781,0.006003,-0.001408,0.004123,0.003771,0.002459,0.009351,-0.003636,0.001179,0.004496
2018-01-09,-0.000115,0.007538,0.001700,-0.002069,-0.018956,-0.037459,0.015393,0.004676,-0.004302,0.001544,...,-0.012007,-0.002519,0.007754,0.006415,-0.002555,-0.001402,-0.009844,0.011773,0.001963,-0.004246


## Generation of Log Returns

Log returns are calculated to support quantitative financial analysis and time series modeling due to their favorable mathematical and statistical properties.


In [96]:
log_returns = np.log(
    adj_close / adj_close.shift(1)
).dropna()

## Price Normalization

Asset prices are normalized to a common starting value for meaningful comparison of price movements across different stocks during exploratory data analysis.


In [97]:
normalized_prices = (
    adj_close / adj_close.iloc[0]
)

normalized_prices.head()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
2018-01-02,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2018-01-03,0.999826,1.015649,1.002211,1.012406,1.017722,1.051913,1.018870,1.012775,1.019781,1.012984,...,1.008723,1.014976,1.005372,1.005386,1.008341,0.999646,0.992144,1.009568,1.004591,1.019640
2018-01-04,1.004470,1.009857,1.000510,1.011298,1.023567,1.103825,1.014576,1.017309,1.007185,1.009454,...,1.009636,1.021103,1.014685,1.012743,1.013438,1.002476,0.983905,1.011004,1.007885,1.021051
2018-01-05,1.015906,1.027436,1.003402,1.015397,1.029412,1.081967,1.020621,1.033751,1.024845,1.021754,...,1.015620,1.020694,1.017550,1.019706,1.024097,1.006897,0.983522,1.019615,1.015870,1.020228
2018-01-08,1.012133,1.010975,1.000510,1.017169,1.054299,1.118397,1.020339,1.048662,1.086699,1.032688,...,1.030632,1.026821,1.016118,1.023910,1.027958,1.009374,0.992719,1.015907,1.017067,1.024815


In [98]:
normalized_prices.to_csv(
    '../data/processed/normalized_prices.csv'
)

## Rolling Volatility Estimation

Rolling 30-day volatility is calculated to analyze changes in asset risk and price variability over time for future financial and stability analysis. A 30-day window is used because it approximately represents one trading month, making it suitable for capturing short-term market risk dynamics.


In [99]:
rolling_volatility = (
    returns.rolling(30).std()
)

rolling_volatility.head()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
rolling_volatility.to_csv(
    '../data/processed/rolling_volatility.csv'
)

## Save Sector Metadata
Sector information is extracted and saved to support sector-wise analysis, asset grouping, and future cointegration and relationship studies between economically related companies.

In [101]:
sector_info = sp500[
    ['Symbol', 'GICS Sector']
]

sector_info['Symbol'] = (
    sector_info['Symbol']
    .str.replace('.', '-', regex=False)
)

sector_info.to_csv(
    '../data/processed/sector_info.csv',
    index=False
)

sector_info.head()

,Symbol,GICS Sector
0,MMM,Industrials
1,AOS,Industrials
2,ABT,Health Care
3,ABBV,Health Care
4,ACN,Information Technology


In [102]:
returns.to_csv(
    '../data/processed/returns.csv'
)

log_returns.to_csv(
    '../data/processed/log_returns.csv'
)

In [103]:
returns.describe()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
count,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,...,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000
mean,0.001155,0.000691,0.000375,0.001032,0.001461,0.002445,0.000556,0.000956,0.001553,0.001117,...,0.000772,0.000574,0.000470,0.000547,0.001005,0.000343,0.000458,0.000406,0.000571,0.000620
std,0.019208,0.016816,0.015551,0.021625,0.028339,0.035360,0.016116,0.021552,0.029079,0.018724,...,0.014170,0.019825,0.014687,0.013404,0.016522,0.009804,0.012751,0.010975,0.014899,0.018999
min,-0.128647,-0.162524,-0.100389,-0.166149,-0.203576,-0.173144,-0.082608,-0.140494,-0.242344,-0.138752,...,-0.113757,-0.201412,-0.137093,-0.113441,-0.138140,-0.093956,-0.113577,-0.098610,-0.126686,-0.122248
25%,-0.007796,-0.007053,-0.007047,-0.010223,-0.013327,-0.016805,-0.007283,-0.010302,-0.012422,-0.007663,...,-0.005918,-0.008881,-0.005891,-0.005542,-0.007084,-0.004275,-0.005873,-0.004970,-0.006399,-0.009472
50%,0.001177,0.001127,0.000680,0.001195,0.001367,0.000683,0.000351,0.001175,0.001970,0.001518,...,0.000886,0.001023,0.000842,0.000967,0.001740,0.000555,0.001021,0.000574,0.001343,0.000583
75%,0.010921,0.009183,0.008565,0.012575,0.017135,0.020610,0.008498,0.012330,0.016383,0.010383,...,0.007559,0.010246,0.007502,0.007316,0.009801,0.005359,0.006960,0.006031,0.008447,0.010765
max,0.153289,0.137673,0.109360,0.183876,0.161058,0.238205,0.118180,0.135359,0.203882,0.101715,...,0.117085,0.160373,0.131566,0.126512,0.134257,0.085106,0.127934,0.077057,0.108881,0.126868


In [104]:
returns.max()

returns.min()

Ticker
AAPL   -0.128647
ABBV   -0.162524
ABT    -0.100389
ADI    -0.166149
AMAT   -0.203576
          ...   
XLP    -0.093956
XLU    -0.113577
XLV    -0.098610
XLY    -0.126686
XOM    -0.122248
Length: 108, dtype: float64

## Dataset summary

In [105]:
print("FINAL DATASET SUMMARY")

print("-" * 40)

print(
    "Number of Assets:",
    adj_close.shape[1]
)

print(
    "Trading Days:",
    adj_close.shape[0]
)

print(
    "Start Date:",
    adj_close.index.min()
)

print(
    "End Date:",
    adj_close.index.max()
)

print(
    "Remaining Missing Values:",
    adj_close.isnull().sum().sum()
)

FINAL DATASET SUMMARY
----------------------------------------
Number of Assets: 108
Trading Days: 2112
Start Date: 2018-01-02 00:00:00
End Date: 2026-05-28 00:00:00
Remaining Missing Values: 0
